# Using Tools with Language Models

John Martinez

In [1]:
import json

catalog = {
    "gloves": {"stock": 7, "unit_cents": 325},
    "goggles": {"stock": 4, "unit_cents": 850},
}

for item, fields in catalog.items():
    print(item, fields)

print("Items:", len(catalog))

gloves {'stock': 7, 'unit_cents': 325}
goggles {'stock': 4, 'unit_cents': 850}
Items: 2


### Step 1 Question

A successful lookup provides current evidence from the catalog, while an answer from memory may be outdated or incorrect.

def lookup_stock(item):
    if item not in catalog:
        return None

    return {
        "item": item,
        "stock": catalog[item]["stock"]
    }


print(lookup_stock("gloves"))
print(lookup_stock("goggles"))
print(lookup_stock("boots"))

In [3]:
def supply_quote(item, units):

    if item not in catalog:
        return {"error": "unknown item"}

    return {
        "item": item,
        "units": units,
        "total_cents": units * catalog[item]["unit_cents"]
    }


print(supply_quote("gloves", 3))
print(supply_quote("goggles", 2))

{'item': 'gloves', 'units': 3, 'total_cents': 975}
{'item': 'goggles', 'units': 2, 'total_cents': 1700}


## Tool Functions

The lookup tool retrieves information from the inventory. The quote tool calculates the cost of requested items using the current catalog data.

In [5]:
TOOLS = {
    "lookup_stock": lookup_stock,
    "supply_quote": supply_quote,
}

print(list(TOOLS))


['lookup_stock', 'supply_quote']


In [6]:
def dispatch_tool(name, arguments):

    if name not in TOOLS:
        return {"error": "tool not allowed"}

    return TOOLS[name](**arguments)


print(
    dispatch_tool(
        "lookup_stock",
        {"item": "gloves"}
    )
)

print(
    dispatch_tool(
        "supply_quote",
        {"item": "goggles", "units": 3}
    )
)

{'item': 'gloves', 'stock': 7}
{'item': 'goggles', 'units': 3, 'total_cents': 2550}


## Tool Dispatch

The dispatch function allows only approved tools to run. This prevents calls to tools that are not on the allowlist.

In [7]:
print(
    dispatch_tool(
        "unknown_tool",
        {}
    )
)

{'error': 'tool not allowed'}


In [8]:
request = {
    "tool": "lookup_stock",
    "arguments": {
        "item": "goggles"
    }
}

result = dispatch_tool(
    request["tool"],
    request["arguments"]
)

print(result)

{'item': 'goggles', 'stock': 4}


In [9]:
request = {
    "tool": "supply_quote",
    "arguments": {
        "item": "gloves",
        "units": 6
    }
}

result = dispatch_tool(
    request["tool"],
    request["arguments"]
)

print(result)

{'item': 'gloves', 'units': 6, 'total_cents': 1950}


## Structured Requests

Structured requests separate the tool name from its arguments. This makes tool use more predictable and easier to validate.

In [10]:
tool_calls = [
    {
        "tool": "lookup_stock",
        "arguments": {"item": "gloves"}
    },
    {
        "tool": "supply_quote",
        "arguments": {"item": "goggles", "units": 2}
    }
]

for call in tool_calls:
    print(
        dispatch_tool(
            call["tool"],
            call["arguments"]
        )
    )

{'item': 'gloves', 'stock': 7}
{'item': 'goggles', 'units': 2, 'total_cents': 1700}


In [11]:
request_text = """
Please check whether goggles are available.
"""

print(request_text.strip())

Please check whether goggles are available.


In [12]:
tool_request = {
    "tool": "lookup_stock",
    "arguments": {
        "item": "goggles"
    }
}

result = dispatch_tool(
    tool_request["tool"],
    tool_request["arguments"]
)

print("Tool result:")
print(result)


Tool result:
{'item': 'goggles', 'stock': 4}


## Observation

The tool result provides evidence from the current inventory rather than relying on a generated answer.

In [13]:
history = []

history.append(
    {
        "tool": "lookup_stock",
        "result": lookup_stock("gloves")
    }
)

history.append(
    {
        "tool": "supply_quote",
        "result": supply_quote("gloves", 6)
    }
)

for item in history:
    print(item)

{'tool': 'lookup_stock', 'result': {'item': 'gloves', 'stock': 7}}
{'tool': 'supply_quote', 'result': {'item': 'gloves', 'units': 6, 'total_cents': 1950}}


In [14]:
approved = True

request = {
    "tool": "supply_quote",
    "arguments": {
        "item": "gloves",
        "units": 4
    }
}

if approved:
    result = dispatch_tool(
        request["tool"],
        request["arguments"]
    )
else:
    result = {"status": "blocked"}

print(result)

{'item': 'gloves', 'units': 4, 'total_cents': 1300}


In [15]:
approved = False

request = {
    "tool": "supply_quote",
    "arguments": {
        "item": "gloves",
        "units": 4
    }
}

if approved:
    result = dispatch_tool(
        request["tool"],
        request["arguments"]
    )
else:
    result = {"status": "blocked"}

print(result)

{'status': 'blocked'}


## Human Approval

A tool call may require approval before execution. If approval is denied, no action is performed.

In [16]:
records = []

approved = False

if approved:
    records.append("quote created")

print("Records:", records)

Records: []


In [17]:
summary = {
    "catalog_items": len(catalog),
    "tools": len(TOOLS),
    "history_entries": len(history)
}

for key, value in summary.items():
    print(key, ":", value)

catalog_items : 2
tools : 2
history_entries : 2


In [18]:
print("Available tools:")

for tool_name in TOOLS:
    print("-", tool_name)

Available tools:
- lookup_stock
- supply_quote


# Reflection

This activity demonstrated how tools can provide reliable information from external data sources. By using tool calls, a language model can retrieve current information and perform calculations instead of relying only on generated text. Tool allowlists, validation, and approval steps help ensure safe and controlled execution.